# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [2]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [3]:
n_missing_study = students["study_hours_reported"].isna().sum()
print(f"missing: {n_missing_study} | present: {students['study_hours_reported'].notna().sum()}")

missing: 255 | present: 1772


**Your justification (1–2 sentences):** study_hours_reported is roughly symmetric with no extreme skew (mean ~10.5, std ~3.9), so I'd use **mean imputation** — filling the 255 blanks with the column mean rather than dropping rows, since dropping would cost about 13% of the sample and the values look missing-at-random rather than tied to any other variable.

In [4]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [5]:
housing_map = {
    "on-campus": "On-Campus", "off-campus": "Off-Campus", "off campus": "Off-Campus",
    "with family": "With Family",
}
students["housing_clean"] = students["housing"].str.strip().str.lower().map(housing_map)
print("before:", students["housing"].value_counts().to_dict())
print("after:", students["housing_clean"].value_counts().to_dict())

before: {'Off-Campus': 755, 'On-Campus': 480, 'With Family': 420, 'off campus': 113, 'Off-campus': 77, 'with family': 72, 'on-campus': 69, ' On-Campus': 41}
after: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


In [6]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [7]:
age_mask = (students["age"] < 16) | (students["age"] > 100)
impossible_ages = sorted(students.loc[age_mask, "age"].unique())
n_neg_work = (students["work_hours_per_week"] < 0).sum()

# Fix: an impossible age (-22, 0, 1, 199, 220...) can't be recovered, so it becomes missing.
students.loc[age_mask, "age"] = np.nan

# Fix: negative work hours look like a sign error, not nonsense -- the magnitude is
# still plausible (e.g. -12 -> 12), so we recover it instead of discarding it.
students["work_hours_per_week"] = students["work_hours_per_week"].abs()

print("impossible ages found:", impossible_ages)
print("negative work-hour rows found:", n_neg_work)
print("ages now missing:", students["age"].isna().sum())
print("work hours still negative:", (students["work_hours_per_week"] < 0).sum())

impossible ages found: [np.int64(-22), np.int64(0), np.int64(1), np.int64(3), np.int64(199), np.int64(220)]
negative work-hour rows found: 4
ages now missing: 6
work hours still negative: 0


In [8]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [9]:
students_dedup = students.drop_duplicates(subset="student_id", keep="first")
print(students_dedup.shape, students_dedup["student_id"].is_unique)

(2000, 13) True


**Why not de-dupe enroll / activity? (1 sentence):** Each row there is a distinct event (one course enrollment, one week of activity) rather than a snapshot of the student, so a repeated student_id is expected and dropping those rows would erase real enrollments and real weeks of data.

In [10]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [11]:
students_dedup = students_dedup.copy()
students_dedup["student_id"] = students_dedup["student_id"].str.replace("NU-", "", regex=False).astype(int)

# per-student activity summary
activity_summary = activity.copy()
activity_summary["student_id"] = activity_summary["student_id"].str.replace("NU-", "", regex=False).astype(int)
activity_summary = activity_summary.groupby("student_id")["minutes_active"].sum().reset_index(name="total_minutes_active")

# per-student course-enrollment summary
grade_points = {"A": 4, "B": 3, "C": 2, "D": 1, "F": 0, "W": np.nan}  # W (withdraw) has no grade point
enroll_summary = enroll.copy()
enroll_summary["grade_points"] = enroll_summary["grade"].map(grade_points)
enroll_summary = enroll_summary.groupby("sid").agg(
    n_courses=("course", "count"),
    avg_grade_points=("grade_points", "mean"),
).reset_index().rename(columns={"sid": "student_id"})

analysis = (
    students_dedup
    .merge(activity_summary, on="student_id", how="left")
    .merge(enroll_summary, on="student_id", how="left")
)
print(analysis.shape, analysis["student_id"].is_unique)

(2000, 16) True


In [12]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [13]:
student_keys = set(analysis["student_id"])
matched = enroll["sid"].isin(student_keys).sum()
orphans = len(enroll) - matched
print(f"matched: {matched} | orphans: {orphans} | total enroll rows: {len(enroll)}")

matched: 8041 | orphans: 47 | total enroll rows: 8088


In [14]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [15]:
before_nat = pd.to_datetime(students["enrollment_date"], errors="coerce").isna().sum()
dates_parsed = pd.to_datetime(students["enrollment_date"], format="mixed", errors="coerce")
students["enrollment_date_parsed"] = dates_parsed
print(f"NaT with naive parse: {before_nat} | NaT with mixed-format parse: {dates_parsed.isna().sum()}")

NaT with naive parse: 1470 | NaT with mixed-format parse: 0


In [17]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [18]:
# bring the parsed date onto the analysis table (join back on the original NU- id)
date_lookup = students[["student_id", "enrollment_date_parsed"]].copy()
date_lookup["student_id"] = date_lookup["student_id"].str.replace("NU-", "", regex=False).astype(int)
date_lookup = date_lookup.drop_duplicates(subset="student_id", keep="first")
analysis = analysis.merge(date_lookup, on="student_id", how="left")

before_analysis_nat = analysis["enrollment_date_parsed"].isna().sum()
print("blank enrollment dates in analysis:", before_analysis_nat)

analysis["study_hours_reported"] = analysis["study_hours_reported"].fillna(analysis["study_hours_reported"].mean())
analysis["study_z"] = (analysis["study_hours_reported"] - analysis["study_hours_reported"].mean()) / analysis["study_hours_reported"].std()
analysis["gpa_band"] = pd.cut(analysis["final_gpa"], bins=[-0.01, 1, 2, 3, 4], labels=["0-1", "1-2", "2-3", "3-4"])
print(analysis["gpa_band"].value_counts())
print("study_z mean:", analysis["study_z"].mean())

blank enrollment dates in analysis: 0
gpa_band
2-3    1102
1-2     528
3-4     300
0-1      70
Name: count, dtype: int64
study_z mean: -8.970602038971265e-17


In [19]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [20]:
analysis.to_csv("northgate_clean.csv", index=False)
print("wrote", len(analysis), "rows to northgate_clean.csv")

wrote 2000 rows to northgate_clean.csv


In [21]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- **Missing study hours (255 blanks)** → filled with the column mean. The distribution is roughly symmetric with no heavy skew, so mean imputation won't distort it, and dropping the rows would throw out 13% of the roster for no real benefit.
- **Housing categories** → collapsed 8 messy variants (casing, stray spaces, "off campus" vs "off-campus") into the 3 real groups: On-Campus, Off-Campus, With Family. This was string noise, not disagreement about the underlying categories.
- **Impossible ages (-22, 0, 1, 3, 199, 220)** → set to missing (NaN). There's no way to recover what the real age was from a value like 220 or -22, so treating it as a data-entry error and marking it missing is more honest than guessing or leaving a physically impossible number in the table.
- **Negative work hours (4 rows)** → fixed with `abs()`. Unlike the ages, these look like a sign-flip on an otherwise plausible number (e.g. -12 is very likely a mistyped 12), so the magnitude is worth keeping rather than discarding.
- **Duplicate students (27 pairs, 54 rows)** → kept the first occurrence of each student_id and dropped the rest, since `student_records` is supposed to be one row per person. 18 of the 27 pairs were exact duplicates; the rest differed slightly (likely a re-entered or updated record), so keeping the first consistently avoids picking a value at random.
- **Why not de-dupe enroll/activity** → each row there is a distinct event (one course enrollment, one week of activity), not a snapshot of the student, so a repeated student_id is expected and dropping those rows would erase real enrollments and real weeks of data.
- **Join key** → stripped the "NU-" prefix from `student_id` and cast it to int to match the plain integer `sid`/`student_id` keys in `course_enrollments` and `weekly_activity`.
- **Integration** → merged in a per-student activity summary (total `minutes_active`) and a per-student enrollment summary (course count, average grade points, with withdrawals excluded from the grade average since "W" isn't a grade).
- **Dates** → parsed with `format="mixed"` instead of a single fixed format, since the column mixes `YYYY-MM-DD`, `MM/DD/YYYY`, and `DD-Mon-YYYY` styles. A naive single-format parse loses ~1,470 valid dates to NaT; the mixed parser loses none, and the parsed column was carried into the final `analysis` table.

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [22]:
mean_gpa = analysis["final_gpa"].mean()
corr = analysis["final_gpa"].corr(analysis["total_minutes_active"])
raw_mean_gpa = students["final_gpa"].mean()  # includes the 27 duplicate rows, pre-cleaning

print("mean GPA (clean):", round(mean_gpa, 3))
print("correlation between final_gpa and total_minutes_active:", round(corr, 3))
print()
print("Takeaway: LMS engagement tracks GPA closely (r ≈ 0.80) -- students who log more")
print("minutes_active tend to have meaningfully higher final grades.")
print(f"Cleaning barely moved the mean GPA ({round(raw_mean_gpa,3)} raw vs {round(mean_gpa,3)} clean),")
print("since duplicates and impossible values were a small slice of the data -- the real value of")
print("cleaning was making the join and the activity/enrollment relationship trustworthy, not shifting the GPA number.")

mean GPA (clean): 2.305
correlation between final_gpa and total_minutes_active: 0.796

Takeaway: LMS engagement tracks GPA closely (r ≈ 0.80) -- students who log more
minutes_active tend to have meaningfully higher final grades.
Cleaning barely moved the mean GPA (2.307 raw vs 2.305 clean),
since duplicates and impossible values were a small slice of the data -- the real value of
cleaning was making the join and the activity/enrollment relationship trustworthy, not shifting the GPA number.


In [23]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
